<!-- NB_DOC_INTRO_V1 -->
# Preprocessing — pipeline complet pour ML

> 📚 **Doc thematique** : [docs/01_STRUCTURES.md](docs/01_STRUCTURES.md) (Structures Python / DataFrame / Preprocessing)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Phase obligatoire** avant tout modele ML : nettoyer + transformer les donnees brutes en features prêtes pour sklearn/XGBoost. Ce notebook couvre tout : NaN, encoding (Ordinal/OneHot/Target), scaling (Standard/MinMax/Robust/Quantile), outliers, deséquilibre (SMOTE), feature engineering (PolynomialFeatures, interactions).

**Le tout dans des `Pipeline` + `ColumnTransformer` sklearn** — pattern recommande pour eviter le data leakage.

## Plan

1. Setup + dataset jouet (mix types)
2. Gestion NaN (`SimpleImputer`, `KNNImputer`, `IterativeImputer`)
3. Encoding categorielles (Ordinal, OneHot, Target)
4. Scaling (Standard, MinMax, Robust, Quantile, PowerTransformer)
5. Outliers en pipeline (RobustScaler + clipping)
6. Pipeline + ColumnTransformer (anti-leakage)
7. Deséquilibre (class_weight, SMOTE)
8. Feature engineering (PolynomialFeatures, interactions)
9. Pieges et anti-patterns
10. References


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer,
    PowerTransformer, OneHotEncoder, OrdinalEncoder, PolynomialFeatures,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Dataset jouet — Titanic-like avec NaN

In [ ]:
# Dataset jouet mixte (numerique + categoriel + NaN)
n = 500
df = pd.DataFrame({
    "age":       np.where(np.random.rand(n) > 0.1, np.random.randint(18, 80, n), np.nan),
    "fare":      np.random.lognormal(3, 1, n).round(2),
    "sex":       np.random.choice(["male", "female"], n, p=[0.6, 0.4]),
    "class":     np.random.choice(["first", "second", "third"], n, p=[0.2, 0.3, 0.5]),
    "embarked":  np.where(np.random.rand(n) > 0.05, np.random.choice(["S", "C", "Q"], n), None),
    "n_family":  np.random.poisson(1.5, n),
})
# Target : probabilite de survie depend de class + sex (proxy realiste)
prob_surv = 0.3 + 0.4 * (df["sex"] == "female") - 0.2 * (df["class"] == "third")
df["survived"] = (np.random.rand(n) < prob_surv).astype(int)

print(f"Shape : {df.shape}")
print(f"NaN par col :\n{df.isna().sum()}")
df.head()

## 2. Imputation NaN — `SimpleImputer`, `KNNImputer`

In [ ]:
# SimpleImputer : moyenne, mediane, most_frequent, constant
imp_num = SimpleImputer(strategy="median")
imp_cat = SimpleImputer(strategy="most_frequent")

# KNN imputation (utilise les k voisins similaires)
imp_knn = KNNImputer(n_neighbors=5)

# Demo : imputation mediane sur age
age_imputed = imp_num.fit_transform(df[["age"]])
print(f"age original NaN : {df['age'].isna().sum()}")
print(f"age impute NaN : {pd.isna(age_imputed).sum()}")
print(f"Mediane utilisee : {imp_num.statistics_[0]}")

## 3. Encoding categorielles

In [ ]:
# OrdinalEncoder : assigne un entier par categorie (utile si arbre, mais ATTENTION : ordre arbitraire pour LogReg)
ord_enc = OrdinalEncoder()
class_ord = ord_enc.fit_transform(df[["class"]])
print(f"OrdinalEncoder('class') : {ord_enc.categories_}, exemple : {class_ord[:5].ravel()}")

# OneHotEncoder : 1 colonne binaire par categorie (recommande pour LogReg / NN)
ohe = OneHotEncoder(sparse_output=False, drop="first")  # drop='first' evite multicolinearite
class_ohe = ohe.fit_transform(df[["class"]])
print(f"\nOneHotEncoder('class') : categories={ohe.categories_[0]}, shape={class_ohe.shape}")
print(f"Exemple :\n{class_ohe[:5]}")

# Target encoding : remplace categorie par moyenne de y dans cette categorie
# (en sklearn 1.3+ : TargetEncoder)
try:
    from sklearn.preprocessing import TargetEncoder
    te = TargetEncoder(random_state=SEED, smooth="auto")
    class_te = te.fit_transform(df[["class"]], df["survived"])
    print(f"\nTargetEncoder('class') : moyenne survived par class :")
    for cls, val in zip(te.categories_[0], te.encodings_[0]):
        print(f"  {cls} : {val:.3f}")
except ImportError:
    print("TargetEncoder requires sklearn >= 1.3")

## 4. Scaling — StandardScaler vs MinMax vs Robust vs PowerTransformer

In [ ]:
# StandardScaler : (x - mean) / std → mean=0, std=1
# MinMaxScaler : (x - min) / (max - min) → [0, 1]
# RobustScaler : (x - median) / IQR → robuste aux outliers
# QuantileTransformer : map vers uniform ou normal
# PowerTransformer (Yeo-Johnson / Box-Cox) : rend gaussien

fare = df[["fare"]].values
scalers = {
    "Standard":  StandardScaler().fit_transform(fare),
    "MinMax":    MinMaxScaler().fit_transform(fare),
    "Robust":    RobustScaler().fit_transform(fare),
    "Quantile (normal)": QuantileTransformer(output_distribution="normal", n_quantiles=100, random_state=SEED).fit_transform(fare),
    "PowerTransformer (Yeo-J)": PowerTransformer(method="yeo-johnson").fit_transform(fare),
}

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0, 0].hist(fare, bins=30); axes[0, 0].set_title("Original (lognormal)")
for ax, (name, vals) in zip(axes.flat[1:], scalers.items()):
    ax.hist(vals, bins=30); ax.set_title(name)
plt.tight_layout(); plt.show()

**Quand utiliser quoi ?**
| Scaler | Si... |
|---|---|
| `StandardScaler` | Distribution proche normale, modeles lineaires / SVM / NN |
| `MinMaxScaler` | Image (pixels [0, 255] → [0, 1]), NN avec bornees |
| `RobustScaler` | Outliers presents (utilise mediane + IQR) |
| `QuantileTransformer` | Distribution tres skewed, modele veut uniforme/normal |
| `PowerTransformer` | Pour gaussianiser (utile en regression lineaire) |


## 5. Pipeline + ColumnTransformer — anti-leakage

In [ ]:
# Le pattern STANDARD : ColumnTransformer applique des transformations differentes par colonne,
# puis Pipeline chaine tout. fit() fait fit sur train, transform() applique sur train + test sans refit.

num_cols = ["age", "fare", "n_family"]
cat_cols = ["sex", "class", "embarked"]

preproc = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imp",   SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
    ]), cat_cols),
])

pipe = Pipeline([
    ("preproc", preproc),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
])

X = df.drop(columns="survived")
y = df["survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test, pred):.4f}")
print(classification_report(y_test, pred, target_names=["died", "survived"]))

**Avantage cle** : `pipe.predict(X_test)` applique exactement les memes transformations que sur train (avec la **mediane du train**, la **moyenne du train**, etc.) → **zero data leakage**.

## 6. Outliers — clipping + RobustScaler

In [ ]:
# Clipping = remplacer les valeurs > Q3+3·IQR (ou < Q1-3·IQR) par les bornes
def clip_outliers(s, k=3):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return s.clip(lo, hi)

print(f"fare original : min={df['fare'].min():.1f}, max={df['fare'].max():.1f}")
df["fare_clipped"] = clip_outliers(df["fare"])
print(f"fare clipped  : min={df['fare_clipped'].min():.1f}, max={df['fare_clipped'].max():.1f}")

## 7. Desequilibre — SMOTE

In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline

    # Verifier desequilibre
    print(f"Distribution train :\n{y_train.value_counts()}")

    # SMOTE dans une pipeline (imblearn.Pipeline, pas sklearn.Pipeline !)
    pipe_smote = ImbPipeline([
        ("preproc", preproc),
        ("smote",   SMOTE(random_state=SEED)),
        ("clf",     LogisticRegression(max_iter=1000, random_state=SEED)),
    ])
    pipe_smote.fit(X_train, y_train)
    pred = pipe_smote.predict(X_test)
    print(f"\nAvec SMOTE — Accuracy : {accuracy_score(y_test, pred):.4f}")
    print(classification_report(y_test, pred, target_names=["died", "survived"]))
except ImportError:
    print("imblearn pas installe (uv add --group ml imbalanced-learn)")
    print("Alternative : class_weight='balanced' dans LogisticRegression :")
    pipe_balanced = Pipeline([
        ("preproc", preproc),
        ("clf",     LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
    ])
    pipe_balanced.fit(X_train, y_train)
    pred = pipe_balanced.predict(X_test)
    print(f"class_weight='balanced' — Accuracy : {accuracy_score(y_test, pred):.4f}")

## 8. Feature engineering — PolynomialFeatures + interactions

In [ ]:
# Polynomial features : x1, x2 → x1, x2, x1², x2², x1*x2 (degre=2)
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
X_poly = poly.fit_transform(df[["age", "fare", "n_family"]].fillna(0))
print(f"Poly features : {poly.get_feature_names_out(['age', 'fare', 'n_family'])}")
print(f"Shape : {X_poly.shape}  (vs original {df[['age', 'fare', 'n_family']].shape})")

# interaction_only=True : seulement les croisements (pas x²)
poly_int = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_int = poly_int.fit_transform(df[["age", "fare", "n_family"]].fillna(0))
print(f"\nInteraction only : {poly_int.get_feature_names_out(['age', 'fare', 'n_family'])}")

## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| **Imputer avant le split** (data leakage) | Toujours Pipeline + fit sur train uniquement |
| Scaler sur tout le dataset puis split | Idem : ColumnTransformer dans Pipeline |
| OneHot sans `handle_unknown='ignore'` | Erreur a l'inference si nouvelle categorie |
| `drop_first=True` quand on utilise un arbre | Arbres geren multicolinearite — drop_first inutile |
| `StandardScaler` quand outliers extremes | `RobustScaler` ou clipping d'abord |
| Imputation = moyenne sans questionner | Mean biaise par outliers — preferer median |
| Pas de stratification au train/test split | Desequilibre → `stratify=y` obligatoire |
| Eval seulement par accuracy sur desequilibre | F1, AUC, MCC (cf. DS_Metrics_Reference) |
| Pas tester sur **nouvelles categories** en prod | OHE `handle_unknown='ignore'` + monitoring |


## References

### Documentation
- sklearn preprocessing : https://scikit-learn.org/stable/modules/preprocessing.html
- sklearn pipelines : https://scikit-learn.org/stable/modules/compose.html
- imbalanced-learn : https://imbalanced-learn.org/
- category_encoders : https://contrib.scikit-learn.org/category_encoders/

### Voir aussi
- [Structures_DataFrame.ipynb](Structures_DataFrame.ipynb)
- [Détection D_outliers.ipynb](Détection D_outliers.ipynb)
- [ML_Regression_Classification_CV_GridSearch.ipynb](ML_Regression_Classification_CV_GridSearch.ipynb)
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)
